# Portfolio API — Zerodha Coin (`slug: zerodha_coin`)

Exercises all `/portfolio/*` endpoints scoped to the `zerodha_coin` source.

**Auth pre-req:** Log in to [kite.zerodha.com](https://kite.zerodha.com) inside the AlphaForge Anton Chrome session (`--remote-debugging-port=9299`). Set `ZERODHA_USER_ID` in `backend/.env.cred.local`.

**What one sync fetches:**
- Mutual fund (COIN) holdings via `/api/mf/holdings` — uses the same `enctoken` as the Kite equity source. All rows emitted with `asset_class: mutual_fund`.

In [ ]:
import json, os
from pathlib import Path

SLUG     = "zerodha_coin"
MODE     = "http"          # "in_process" | "http"
BASE     = "http://localhost:8000/api/v1"

AF_USERNAME = os.getenv("AF_USERNAME", "admin")
AF_PASSWORD = os.getenv("AF_PASSWORD", "alphaforge-anton-dev")

if MODE == "in_process":
    from fastapi.testclient import TestClient
    from app.main import app
    client = TestClient(app)
    PREFIX = "/api/v1"
else:
    import httpx
    client = httpx.Client(base_url=BASE, timeout=60.0)
    PREFIX = ""


def _login() -> str:
    r = client.post(
        f"{PREFIX}/auth/token",
        data={"username": AF_USERNAME, "password": AF_PASSWORD},
    )
    if r.status_code != 200:
        raise RuntimeError(
            f"Auth failed ({r.status_code}): {r.text}. "
            "Set AF_USERNAME / AF_PASSWORD env vars if you changed admin creds."
        )
    return r.json()["access_token"]


def _ensure_auth() -> None:
    if "Authorization" not in client.headers:
        client.headers["Authorization"] = f"Bearer {_login()}"


def _request(method: str, path: str, **kw):
    _ensure_auth()
    r = client.request(method, f"{PREFIX}{path}", **kw)
    if r.status_code == 401:
        client.headers["Authorization"] = f"Bearer {_login()}"
        r = client.request(method, f"{PREFIX}{path}", **kw)
    return r.status_code, r.json() if r.headers.get("content-type", "").startswith("application/json") else r.text


def get(path, **kw):  return _request("GET", path, **kw)
def post(path, **kw): return _request("POST", path, **kw)

def pp(obj):
    print(json.dumps(obj, indent=2, default=str))

_ensure_auth()
print(f"Mode: {MODE}  slug: {SLUG}  authed as: {AF_USERNAME}")

## 1. Source info

`status: ready` when `ZERODHA_USER_ID` is set, `unconfigured` otherwise.

In [ ]:
status, body = get(f"/portfolio/sources/{SLUG}")
print(status)
pp(body)

## 2. Sync

Triggers CDP login → enctoken → Kite `/api/mf/holdings` fetch. Enctoken cached in `.cache/brokers/zerodha_coin.json`.

> Requires `MODE="http"` with a live server and an open Chrome session logged in to kite.zerodha.com.

In [ ]:
status, body = post(f"/portfolio/sources/{SLUG}/sync")
print(status, f"  holdings={body.get('holdings_count')}  status={body.get('info', {}).get('status')}")
for h in (body.get("holdings") or [])[:8]:
    print(f"  {h['asset_class']:12} {h['symbol']:35} qty={h['quantity']:<8.3f}  nav=₹{h['last_price']:>10,.2f}  pnl={h['pnl_pct']:>+.1f}%")

## 3. Holdings — zerodha_coin only

In [ ]:
status, body = get("/portfolio/holdings", params={"source": SLUG})
print(status, "  totals:", body.get("totals"))
print(f"\n{len(body.get('holdings', []))} holdings:")
for h in body.get("holdings", []):
    print(f"  [{h['asset_class']:12}] {h['symbol']:35} avg=₹{h['avg_price']:>10,.2f}  nav=₹{h['last_price']:>10,.2f}  pnl={h['pnl_pct']:>+.1f}%")

## 4. Allocation (zerodha_coin)

In [ ]:
status, body = get("/portfolio/holdings", params={"source": SLUG})
print("Allocation:")
for a in body.get("allocation", []):
    print(f"  {a['asset_class']:12} ₹{a['value']:>14,.0f}  ({a['pct']:>5.1f}%)")

## 5. Treemap (zerodha_coin)

In [ ]:
status, body = get("/portfolio/treemap", params={"source": SLUG})
print(status)
for c in (body.get("cells") or [])[:10]:
    print(f"  {c['symbol']:35} {c['pct']:>5.1f}% @ ({c['left_pct']:>5.1f}, {c['top_pct']:>5.1f}) {c['width_pct']:>5.1f}x{c['height_pct']:>5.1f}")

## 6. Rebalance (zerodha_coin)

In [ ]:
status, body = get("/portfolio/rebalance", params={})
print("Drift:")
for d in body.get("drift", []):
    print(f"  {d['asset_class']:12} target {d['target_pct']:>5.1f}% · actual {d['actual_pct']:>5.1f}% · drift {d['drift_pct']:>+5.1f}%")
print("\nSuggestions:")
for s in body.get("suggestions", []):
    print("  -", s["action"])

## 7. Cash (not supported)

`zerodha_coin` does not expose free cash — Coin is an MF platform, not a trading account. `POST /portfolio/cash/zerodha_coin/sync` should return `422`.

In [ ]:
status, body = post(f"/portfolio/cash/{SLUG}/sync")
assert status == 422, f"Expected 422 (supports_cash=False), got {status}: {body}"
print(f"✓ POST /portfolio/cash/{SLUG}/sync → 422 (expected — zerodha_coin does not support cash)")

## 8. Standalone dump

Bypasses FastAPI — tests the helper + on-disk cache write path in isolation.

In [ ]:
import asyncio, sys
sys.path.insert(0, str(Path.cwd().parent))

from app.modules.brokers.zerodha_coin.zerodha_coin_dump import dump_zerodha_coin

path = await dump_zerodha_coin()
print(f"Dumped → {path}")
print(f"Size: {path.stat().st_size:,} bytes")

## 9. Reset cache

In [ ]:
from app.modules.brokers import SOURCES

SOURCES[SLUG].reset()
status, body = get(f"/portfolio/sources/{SLUG}")
print(f"{SLUG}: status={body['status']}  holdings={body['holdings_count']}")